## SETUPS ##

In [1]:
# Core libraries
import pandas as pd
import numpy as np
import re
from datetime import datetime

# The star of the show
from google_play_scraper import app, reviews, Sort

print("Libraries loaded successfully!")

Libraries loaded successfully!


## SCRAPPING ##

In [4]:
CBE_APP_ID = 'com.combanketh.mobilebanking'

# Step 1: Get app metadata (rating, installs, description...)
app_info = app(
    CBE_APP_ID,
    lang='en',    # Language: English
    country='et'  # Country: Ethiopia
)

print("=" * 50)
print("CBE App Info")
print("=" * 50)
print(f"App Title   : {app_info['title']}")
print(f"Current Score: {app_info['score']}")
print(f"Total Ratings: {app_info['ratings']:,}")
print(f"Total Reviews: {app_info['reviews']:,}")
print(f"Installs     : {app_info['installs']}")

CBE App Info
App Title   : Commercial Bank of Ethiopia
Current Score: 4.2866945
Total Ratings: 48,615
Total Reviews: 9,340
Installs     : 10,000,000+


In [6]:
# Step 2: Scrape reviews
print(f"Scraping reviews for CBE...")

result, continuation_token = reviews(
    CBE_APP_ID,
    lang='en',
    country='et',
    sort=Sort.NEWEST,       # Most recent first
    count=400,              # Ask for more than 400 to be safe
    filter_score_with=None  # All star ratings
)

print(f"Collected {len(result)} raw reviews")

Scraping reviews for CBE...
Collected 400 raw reviews


In [7]:
# Let's inspect what a single raw review looks like
print("Keys in a single review:")
print(list(result[0].keys()))

print("\nFirst raw review (sample):")
for key, value in result[0].items():
    print(f"  {key}: {value}")

Keys in a single review:
['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']

First raw review (sample):
  reviewId: 59f65bd1-a5f7-4f88-bb73-6e0a452317a1
  userName: nesira Kemila
  userImage: https://play-lh.googleusercontent.com/a/ACg8ocIxDx0VuYKUE_RTyFQ-hvlxvQxj0fUyQ8mNHWiiCBO8AR62KA=mo
  content: pels
  score: 5
  thumbsUpCount: 0
  reviewCreatedVersion: 5.3.0
  at: 2026-05-18 08:15:35
  replyContent: None
  repliedAt: None
  appVersion: 5.3.0


In [8]:
# Step 3: Extract only the columns we need
raw_data = []

for r in result:
    raw_data.append({
        'review_id': r.get('reviewId', ''),
        'review'   : r.get('content', ''),
        'rating'   : r.get('score', None),
        'date'     : r.get('at', None),
        'bank'     : 'Awash Bank',
        'source'   : 'Google Play'
    })

# Build a DataFrame
df_raw = pd.DataFrame(raw_data)

print(f"Shape: {df_raw.shape}")
df_raw.head()

Shape: (400, 6)


,review_id,review,rating,date,bank,source
0,59f65bd1-a5f7-4f88-bb73-6e0a452317a1,pels,5,2026-05-18 08:15:35,Awash Bank,Google Play
1,0cf149ea-af00-4e00-8458-d2428d928589,What an excellent app with smooth performance !!,5,2026-05-18 07:36:31,Awash Bank,Google Play
2,523d4953-abb1-4f53-b478-51c951aefcb0,በጣም ጥሩ ነው እነማሰግነለን,5,2026-05-18 05:55:46,Awash Bank,Google Play
3,79aecee0-a711-4a4a-96bd-e9836c4afd12,svabst keessatti argamu yoo ta'u yeroo ammaa k...,5,2026-05-18 01:12:03,Awash Bank,Google Play
4,72fd0808-c8f7-435a-8337-0308012763da,nays,5,2026-05-17 18:46:14,Awash Bank,Google Play


In [9]:
# Basic shape and types
print(f"Total reviews collected: {len(df_raw)}")
print(f"\nColumn dtypes:")
print(df_raw.dtypes)

Total reviews collected: 400

Column dtypes:
review_id               str
review                  str
rating                int64
date         datetime64[us]
bank                    str
source                  str
dtype: object


In [10]:
# Rating distribution — what do users think?
print("Rating distribution:")
rating_counts = df_raw['rating'].value_counts().sort_index(ascending=False)
for rating, count in rating_counts.items():
    bar = '█' * (count // 5)
    print(f"  {int(rating)} stars: {count:>4}  {bar}")

Rating distribution:
  5 stars:  276  ███████████████████████████████████████████████████████
  4 stars:   29  █████
  3 stars:   25  █████
  2 stars:    9  █
  1 stars:   61  ████████████


In [11]:
# What does the date column look like right now?
print("Sample date values (raw):")
print(df_raw['date'].head(10).to_string())

print(f"\nDate dtype: {df_raw['date'].dtype}")

Sample date values (raw):
0   2026-05-18 08:15:35
1   2026-05-18 07:36:31
2   2026-05-18 05:55:46
3   2026-05-18 01:12:03
4   2026-05-17 18:46:14
5   2026-05-17 17:03:51
6   2026-05-17 16:12:24
7   2026-05-17 15:55:05
8   2026-05-17 14:33:09
9   2026-05-17 13:23:30

Date dtype: datetime64[us]
